# 📊 TikTok Ads Analysis - Professional Report

**Notebook ini menggunakan package `tiktok_ads/` untuk analisis yang sinkron dengan MCP Server.**

## Fitur:
- ✅ Health Score & Risk Assessment
- ✅ Estimasi Profit Impact
- ✅ Statistical Analysis (Mann-Kendall, Linear Regression, Momentum, CV)
- ✅ Data Quality Labeling
- ✅ HTML Report Generation
- ✅ Rolling Window (Quarterly) Analysis

In [ ]:
# Import dari package tiktok_ads (100% sinkron dengan MCP)
from tiktok_ads import (
    # Data loading
    load_product_names, load_data, summary_stats,
    # Analysis
    analyze_products, get_top_products, get_stop_products, get_monitor_products,
    # Evaluation
    generate_ai_evaluation, calculate_health_score, estimate_profit_impact,
    # Report
    run_full_analysis, run_quarterly_analysis,
    # Period
    get_available_periods, get_rolling_window_periods
)

import pandas as pd
from IPython.display import HTML, display, Markdown

print("✅ Package tiktok_ads imported successfully!")

## 1. Load Data

In [ ]:
# Load data
product_map = load_product_names()
df = load_data()

# Quick summary
stats = summary_stats(df)
print(f"\n📊 SUMMARY:")
print(f"   Total Records: {stats['total_records']:,}")
print(f"   Total Products: {stats['total_products']}")
print(f"   Total Cost: Rp {stats['total_cost']:,.0f}")
print(f"   Total Revenue: Rp {stats['total_revenue']:,.0f}")
print(f"   Total Profit: Rp {stats['total_profit']:,.0f}")
print(f"   Overall ROI: {stats['roi']:.2f}x")
print(f"   Date Range: {stats['date_range']}")

## 2. Available Periods

In [ ]:
# Lihat semua periode yang tersedia
periods = get_available_periods(df, verbose=True)
print(f"\nTotal {len(periods)} periods detected")

# Rolling window (3 bulan terakhir)
window = get_rolling_window_periods(df, n_months=3)
print(f"\n🔄 Rolling Window: {window['quarter_label']}")
print(f"   Periods: {window['n_periods']}")
print(f"   Range: {window['range_label']}")

## 3. Run Analysis (dengan Rolling Window)

In [ ]:
# Analyze products dengan rolling window 3 bulan
products = analyze_products(df, product_map, verbose=True, use_rolling_window=True, window_months=3)

print(f"\n📈 Analysis complete: {len(products)} products")

## 4. AI Evaluation

In [ ]:
# Generate AI evaluation
evaluation = generate_ai_evaluation(products)

# Health Score
health = evaluation['health']
print("=" * 60)
print(f"📊 PORTFOLIO HEALTH SCORE: {health['health_score']}/100 ({health['grade']})")
print(f"   Status: {health['interpretation']}")
print("=" * 60)

# Components
print("\n📈 Health Score Components:")
for name, comp in health['components'].items():
    print(f"   {name}: {comp['score']} (weight: {comp['weight']}, value: {comp['value']})")

# Risk
risk = evaluation['risk']
print(f"\n⚠️ Risk Level: {risk['risk_level']} (Score: {risk['risk_score']}/100)")
print("   Risk Factors:")
for factor in risk['factors']:
    print(f"   - {factor}")

## 5. Profit Impact Estimation

In [ ]:
# Profit impact estimation
impact = evaluation['impact']

print("=" * 60)
print("💹 ESTIMASI PROFIT IMPACT")
print("=" * 60)
print(f"\n📊 Current Profit: Rp {impact['current']['profit']:,.0f}")
print(f"📈 Projected Profit: Rp {impact['projected']['profit']:,.0f}")
print(f"🚀 Growth: +{impact['projected']['growth_pct']:.1f}%")

print("\n📋 Breakdown:")
breakdown = impact['breakdown']
print(f"   Savings dari Stop: +Rp {breakdown['savings_from_stop']:,.0f}")
print(f"   Scale-up Gains: +Rp {breakdown['scale_up_gains']:,.0f}")
print(f"   Reallocation: +Rp {breakdown['reallocation_potential']:,.0f}")
print(f"   Budget Freed: Rp {breakdown['cost_freed']:,.0f}")

print("\n📝 Assumptions:")
for k, v in impact['assumptions'].items():
    print(f"   {k}: {v}")

## 6. Distribution & Top/Stop Products

In [ ]:
# Distribution
dist = evaluation['distribution']
print("\n📈 Distribusi Rekomendasi:")
print(f"   🟢 LANJUTKAN: {dist['lanjutkan']}")
print(f"   🟡 PANTAU: {dist['pantau']}")
print(f"   🔴 HENTIKAN: {dist['hentikan']}")

# Top products
top = get_top_products(products, n=10)
print(f"\n🏆 TOP {len(top)} PRODUCTS (Scale Up):")
for i, p in enumerate(top[:5], 1):
    print(f"   {i}. {p['product_name'][:40]}")
    print(f"      ROI: {p['roi']:.1f}x | Profit: Rp {p['profit']:,.0f} | Score: {p['score']:.1f}")

# Stop products
stop = get_stop_products(products)
print(f"\n🚫 STOP PRODUCTS ({len(stop)}):")
for i, p in enumerate(stop[:5], 1):
    loss = abs(p['profit']) if p['profit'] < 0 else 0
    print(f"   {i}. {p['product_name'][:40]}")
    print(f"      ROI: {p['roi']:.2f}x | Loss: Rp {loss:,.0f}")

## 7. Strategic Insights

In [ ]:
# Strategic insights
print("\n🎯 STRATEGIC INSIGHTS:")
print("=" * 60)
for i, insight in enumerate(evaluation['strategic_insights'], 1):
    print(f"\n{i}. [{insight['type']}]")
    print(f"   Temuan: {insight['insight']}")
    print(f"   Aksi: {insight['action']}")
    print(f"   Metodologi: {insight['methodology']}")

# Creative comparison
cc = evaluation['creative_comparison']
print("\n📱 Creative Type Comparison:")
print(f"   Video: {cc['video']['count']} products, ROI {cc['video']['roi']:.2f}x")
print(f"   Kartu: {cc['kartu']['count']} products, ROI {cc['kartu']['roi']:.2f}x")

## 8. Generate Full Report (HTML)

In [ ]:
# Generate full quarterly report
print("\n📝 Generating quarterly report...")
result = run_quarterly_analysis(verbose=True)

print(f"\n✅ Report generated!")
print(f"   Markdown: {result['md_path']}")
print(f"   HTML: {result['html_path']}")

# Display link
from IPython.display import FileLink
display(FileLink(result['html_path']))

## 9. Quick Reference: Data Quality

In [ ]:
# Data quality distribution
high = len([p for p in products if '✅' in p['data_quality']])
medium = len([p for p in products if '🔶' in p['data_quality']])
low = len([p for p in products if '⚠️' in p['data_quality']])
insufficient = len([p for p in products if '❌' in p['data_quality']])

print("\n📊 Data Quality Distribution:")
print(f"   ✅ HIGH: {high} ({high/len(products)*100:.0f}%)")
print(f"   🔶 MEDIUM: {medium} ({medium/len(products)*100:.0f}%)")
print(f"   ⚠️ LOW: {low} ({low/len(products)*100:.0f}%)")
print(f"   ❌ INSUFFICIENT: {insufficient} ({insufficient/len(products)*100:.0f}%)")

print("\n⚠️ Products with limited data (recommendations tentative):")
limited = [p for p in products if '⚠️' in p['data_quality'] or '❌' in p['data_quality']]
for p in limited[:5]:
    print(f"   - {p['product_name'][:40]} ({p['n_periods']} periods)")

---

## Methodology Reference

| Method | Purpose | Formula |
|--------|---------|--------|
| **Mann-Kendall** | Trend detection | S = Σ sign(xⱼ - xᵢ) |
| **Linear Regression** | Slope & R² | y = mx + b |
| **Momentum** | Recent vs Historical | 70/30 split |
| **CV** | Consistency | CV = (σ/μ) × 100% |
| **Composite Score** | Overall rating | Weighted sum |
| **Health Score** | Portfolio health | 4-component weighted |

---
*This notebook is synchronized with MCP Server `mcp-servers/tiktok-ads/`*